### TF-IDF 방식 계산법

In [1]:
docs = [
    '영화가 너무 재미있었다', 
    '영화가 너무 지루하다', 
    '배우의 연기가 너무 좋았다'
]

In [2]:
tokens = [ doc.split() for doc in docs ]

In [3]:
tokens

[['영화가', '너무', '재미있었다'], ['영화가', '너무', '지루하다'], ['배우의', '연기가', '너무', '좋았다']]

In [4]:
# 단어 사전을 생성 -> 1차원으로 데이터를 변경하고 중복 데이터를 제거 
vocab1 =  []

for token in tokens :
    for word in token:
        vocab1.append(word)
# 리스트에서 중복 값을 제거  -> 집합의 형태로 변경했다가 다시 리스트로 변환 
vocab1 = list( set(vocab1) )
vocab1

['연기가', '배우의', '재미있었다', '좋았다', '영화가', '지루하다', '너무']

In [9]:
vocab = list(set(sum(tokens, [])))

In [11]:
# TF, IDF 계산
import math

In [12]:
# 단어 사전의 길이 
V = len(vocab)
# 전체 문서의 길이 
N = len(docs)

In [14]:
tokens

[['영화가', '너무', '재미있었다'], ['영화가', '너무', '지루하다'], ['배우의', '연기가', '너무', '좋았다']]

In [13]:
# 단어의 개수를 생성 
word_cnt = {
    w :  sum(1 for doc in tokens if w in doc) for w in vocab
}
word_cnt

{'연기가': 1, '배우의': 1, '재미있었다': 1, '좋았다': 1, '영화가': 2, '지루하다': 1, '너무': 3}

In [27]:
# TF 계산식 함수 
def tf(word, doc):
    # word : 단어 사전의 각 원소들 대입
    # doc : tokens의 각 원소들 대입 
    result = math.log( doc.count(word) / len(doc) + 1 )
    # doc.count(word) : 문장에서 특정 단어의 개수 
    # len(doc) : 문장의 단어의 개수
    return result

In [28]:
# IDF 계산 함수 
def idf(word):
    # word : 단어 사전의 각 원소 
    result = math.log( (N) / (word_cnt[word] + 1) )
    # N : docs의 길이 -> 문장들의 개수 
    # word_cnt[word] : 전체 문서에서 특정 단어의 개수 
    return result

In [29]:
X_tfidf = [
    [  tf(w, doc) * idf(w) for w in vocab]  for doc in tokens
]

In [30]:
X_tfidf

[[0.0, 0.0, 0.1166450426074421, 0.0, 0.0, 0.0, -0.0827609748101517],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.1166450426074421, -0.0827609748101517],
 [0.09047692415725579,
  0.09047692415725579,
  0.0,
  0.09047692415725579,
  0.0,
  0.0,
  -0.06419439929632219]]

In [31]:
import pandas as pd 

In [32]:
pd.DataFrame(X_tfidf, columns = vocab)

,연기가,배우의,재미있었다,좋았다,영화가,지루하다,너무
0,0.000000,0.000000,0.116645,0.000000,0.0,0.000000,-0.082761
1,0.000000,0.000000,0.000000,0.000000,0.0,0.116645,-0.082761
2,0.090477,0.090477,0.000000,0.090477,0.0,0.000000,-0.064194


In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [34]:
vec = TfidfVectorizer(
    ngram_range= (1, 1), 
    min_df=1
)

In [35]:
X = vec.fit_transform(docs)

In [36]:
pd.DataFrame(X.toarray(), columns = vec.get_feature_names_out())

,너무,배우의,연기가,영화가,재미있었다,좋았다,지루하다
0,0.425441,0.000000,0.000000,0.547832,0.720333,0.000000,0.000000
1,0.425441,0.000000,0.000000,0.547832,0.000000,0.000000,0.720333
2,0.322745,0.546454,0.546454,0.000000,0.000000,0.546454,0.000000


### LSA(잠재 의미 분석)
- 문서 안에서 단어 사이의 잠재적인 의미 구조를 추출하는 기법 
- TF-IDF 방식은 단어 간의 의미적 유사성을 반영X
- TF-IDF 방식에서 SVD 분해를 하여 단어 간의 의미를 파악 
- 차원 축소를 통해서 관계성을 확인 
- LSA 효과 
    - 벡터 공간의 차원을 줄여서 계산 효율의 증가(차원 축소)
    - '영화', '필름' 비슷한 문맥의 단어를 가까운 벡터로 이동(의미 유추)
    - 문서들을 주제별로 분류 가능 (토픽 분석)
- TruncatedSVD(차원 축소 모델)
    - 절단된 특이 값을 분해 
    - 고차원 희소행렬(값이 0인 행렬)을 낮은 차원으로 압축하여 데이터의 구조적 의미를 유지 
    - 자연어 처리, 추천 시스템, 의미 분석, 잠재적인 토픽 분석에서 주로 사용 
    - TF-IDF 행렬은 고차원 -> 저차원 
    - 0으로 이루어진 희소행렬들을 구조적인 의미를 유지하면서 값들을 부여 
    - 같은 토픽의 문서는 같은 벡터 공간에서 가깝게 위치 -> 유사도 기반 자연어 처리에서 활용 

In [37]:
from sklearn.decomposition import TruncatedSVD
from konlpy.tag import Okt

In [38]:
okt = Okt()
def tokenize(text):
    result = []
    for word, pos in okt.pos(text):
        if pos in ['Noun', 'Adjective', 'Verb']:
            result.append(word)
    return result

In [39]:
docs = [
    '이 영화가 정말 재미있었다', 
    '매우 연기가 뛰어나다', 
    '이 영화 별로다', 
    '지루한 영화는 보기 어렵다', 
    '정말 훌룡한 연기였다', 
    '연기가 별로라서 지루했다'
]

In [40]:
tfidf = TfidfVectorizer(
    tokenizer=tokenize, 
    ngram_range= (1,1), 
    min_df=1, 
    max_df=0.8, 
    lowercase=False
)

In [41]:
X_tfidf = tfidf.fit_transform(docs)

c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [42]:
X_tfidf.shape

(6, 14)

In [43]:
# 차원 축소 
lsa = TruncatedSVD(n_components=2, random_state=42)
X_lsa = lsa.fit_transform(X_tfidf)

In [45]:
df_lsa = pd.DataFrame(X_lsa, columns = ['topic1', 'topic2'])
df_lsa

,topic1,topic2
0,0.722615,-0.336336
1,0.229990,0.678032
2,0.801574,-0.279071
3,0.346712,-0.383244
4,0.392081,0.481308
5,0.516723,0.493418


In [46]:
df_lsa['document'] = docs
df_lsa

,topic1,topic2,document
0,0.722615,-0.336336,이 영화가 정말 재미있었다
1,0.229990,0.678032,매우 연기가 뛰어나다
2,0.801574,-0.279071,이 영화 별로다
3,0.346712,-0.383244,지루한 영화는 보기 어렵다
4,0.392081,0.481308,정말 훌룡한 연기였다
5,0.516723,0.493418,연기가 별로라서 지루했다


In [47]:
features = tfidf.get_feature_names_out()
features

array(['뛰어나다', '매우', '별로', '보기', '어렵다', '연기', '였다', '영화', '이', '재미있었다',
       '정말', '지루한', '지루했다', '훌룡'], dtype=object)

In [48]:
components = lsa.components_
components

array([[ 0.0830607 ,  0.0830607 ,  0.44101117,  0.10569975,  0.10569975,
         0.28313222,  0.12558933,  0.47611211,  0.47725902,  0.24451986,
         0.30349483,  0.10569975,  0.20031593,  0.12558933],
       [ 0.33833856,  0.33833856,  0.0835962 , -0.161434  , -0.161434  ,
         0.56468446,  0.21301717, -0.33302649, -0.26207738, -0.15725175,
         0.04572844, -0.161434  ,  0.26429403,  0.21301717]])

In [49]:
components.shape

(2, 14)

In [52]:
pd.DataFrame(components,index=['topic1', 'topic2'] ,columns = features).T

,topic1,topic2
뛰어나다,0.083061,0.338339
매우,0.083061,0.338339
별로,0.441011,0.083596
보기,0.105700,-0.161434
어렵다,0.105700,-0.161434
연기,0.283132,0.564684
였다,0.125589,0.213017
영화,0.476112,-0.333026
이,0.477259,-0.262077
재미있었다,0.244520,-0.157252
